In [1]:
import sys

In [2]:
%%capture
try:
    # Attempt to import a module that's only available in Colab
    from google.colab import drive
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    # Colab specific setup
    drive.mount('/content/drive')
    sys.path.append('/content/drive/MyDrive/structure-loss-classification/')
    my_local_data = '/content/drive/MyDrive/types/'
    %cd '/content/drive/MyDrive/structure-loss-classification/'
    %pip install scienceplots
    %pip install pytorch_lightning
else:
    # Local machine setup
    my_local_data = "/mnt/g/My Drive/types/"
    my_local_data_struct = "/mnt/g/My Drive/types-struct/"

In [3]:
import torch
import torch.nn as nn

import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision.models.feature_extraction import (
    get_graph_node_names,
    create_feature_extractor,
)

In [4]:
from sklearn.model_selection import train_test_split, StratifiedKFold

In [5]:
import pytorch_lightning as pl

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(["science", "notebook", "grid"])

In [7]:
import pickle

In [8]:
from models.models import LeNet5
from lightning_modules.lightning_modules import LitLeNet5
from visualization.filters import display_filters
from datasets.data_modules import CustomImageDataModule
from train.train import get_features, train_model

In [9]:
toTensorAndNormalize = transforms.Compose(
    [
        transforms.Resize((244, 244)),
        # transforms.RandomHorizontalFlip(),
        # transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # mean  # std
    ]
)
aux_data = datasets.ImageFolder(root=my_local_data,
                                transform=toTensorAndNormalize)

In [10]:
from datasets.datasets import CustomImageFolder

In [11]:
aux_data_struct = CustomImageFolder(root=my_local_data_struct,
                                   classification_mode='binary',
                                   transform=toTensorAndNormalize)

In [12]:
'binary' in ['binary']

True

In [13]:
# Try to load cached targets first
try:
    with open(f"logdir/cached_targets_struct.pkl", "rb") as f:
        targets = pickle.load(f)
except FileNotFoundError:
    targets = [t for _, t in aux_data_struct]
    # Cache the targets for next time
    with open(f"logdir/cached_targets_struct.pkl", "wb") as f:
        pickle.dump(targets, f)

In [14]:
model2 = LitLeNet5(num_classes=2, learning_rate=0.001)

In [15]:
dataset = datasets.ImageFolder(my_local_data_struct)
print(f"Number of classes: {len(dataset.classes)}")
print(f"Class to index mapping: {dataset.class_to_idx}")

Number of classes: 2
Class to index mapping: {'badIngots': 0, 'goodIngots': 1}


In [16]:
kfold = StratifiedKFold(
    n_splits=4,
    shuffle=True,
)

In [17]:
validation_metrics = []

In [18]:
trainer_config = {
    "patience": 3,
    "accelerator": "gpu",
    "devices": -1,
    "max_epochs": 10,
    "precision": 32,
    "n_steps": 5,
}

In [19]:
# Assuming aux_data is a dataset object and targets are the labels
train_idx, val_idx, _, _ = train_test_split(
    range(len(aux_data_struct)), targets, test_size=0.2, random_state=42
)

train_data = torch.utils.data.Subset(aux_data_struct, train_idx)
val_data = torch.utils.data.Subset(aux_data_struct, val_idx)

In [20]:
data_module = CustomImageDataModule(
    train_dataset=train_data,
    val_dataset=val_data,
    batch_size=16,
    num_workers=4,
)

In [21]:
# Assuming model2 is initialized, and trainer_config is defined
val_metrics = train_model(
    model=model2,
    trainer_config=trainer_config,
    save_dir="logdir-struct/",
    data_module=data_module,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: logdir-struct//LitLeNet5/lightning_logs
/home/alfredosg/.envs/sl-ml/lib/python3.8/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:639: Checkpoint directory logdir-struct/ exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type               | Params
------------------------------------------------
0 | accuracy | MulticlassAccuracy | 0     
1 | f1_score | MulticlassF1Score  | 0     
2 | loss_fn  | CrossEntropyLoss   | 0     
3 | model    | LeNet5             | 29.4 M
------------------------------------------------
29.4 M    Trainable params
0         Non-trainable params
29.4 M    Total params
117.782   Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

/home/alfredosg/.envs/sl-ml/lib/python3.8/site-packages/pytorch_lightning/trainer/call.py:54: Detected KeyboardInterrupt, attempting graceful shutdown...
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      val_accuracy          0.3137255012989044
      val_f1_score          0.3137255012989044
        val_loss            0.7320892214775085
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Fold None Validation Metrics: {'val_loss': tensor(0.7321), 'val_accuracy': tensor(0.3137), 'val_f1_score': tensor(0.3137)}
